In [ ]:
""" Created on April 24, 2026 // @author: Sarah Shi """

import numpy as np
import mineralML as mm

import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'png'

# Interactive Tools for Mapped EDS Data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SarahShi/mineralML/blob/main/docs/examples/mineralML_interactive.ipynb)

This notebook demonstrates **interactive tools for exploring mineralML phase maps**. These tools require a live Python kernel and an interactive Matplotlib backend — they cannot be run on ReadTheDocs. Use the Colab badge above or run locally with `%matplotlib widget` (requires `ipympl`).

```bash
pip install ipympl
```

The tools shown here are:

- **`mm.interactive_pixels`** — click pixels to sample oxide compositions
- **`mm.interactive_line_profile`** — click to draw transects and extract line profiles
- **`mm.plot_locations`** — plot the locations of picked pixels or transects on a map

## 1. Load and run a map

First, run `mm.run_map` as usual to get the result dictionary. See the [EDS mapping notebook](mineralML_mapping.ipynb) for full details on loading and preparing data.

In [ ]:
%matplotlib inline

result = mm.run_map(
    "Maps/09g3",
    renormalize=True,
    units="element_wt%",
    pixel_size_um=4.0,
    scalebar_um=50,
    total_threshold=50,
    min_frac=0.01,
)

## 2. Interactive pixel composition picker

`mm.interactive_pixels` displays the phase map and records the full oxide composition of each pixel you click. Each click:
- Places a marker on the map
- Prints the oxide values below the figure
- Appends a row to the dataframe in `controller["picks"]`

**Keybindings:** `r`/`u` undo last pick &nbsp;|&nbsp; `c` clear all &nbsp;|&nbsp; `q`/Esc quit

After quitting, access your picks as a DataFrame:
```python
controller["picks"]
```

In [ ]:
controller = mm.interactive_pixels(result)

In [ ]:
%matplotlib inline

controller["picks"]

### 2a. Filter to a specific phase

Pass `phase` to restrict the display and clicks to one or more mineral phases. Pixels outside the selected phase are shown as white background.

In [ ]:
%matplotlib widget

controller_opx = mm.interactive_pixels(result, 
                                       phase="Orthopyroxene", # plot only pixels classified as orthopyroxene
                                       )

### 2b. Display an oxide heatmap as background

Pass `oxide_key` to show an oxide or component concentration map instead of the phase map. Combine with `phase` to restrict to a single mineral.

In [ ]:
%matplotlib widget

controller_sio2 = mm.interactive_pixels(result, oxide_key="SiO2", phase="Orthopyroxene")

## 3. Plot pick locations

`mm.plot_locations` overlays picked pixel locations on a map. Pass `map_key` to use an oxide map as the background.

In [ ]:
%matplotlib inline

fig, ax = mm.plot_locations(result, 
                            controller["picks"], # plot only the pixels that have been picked
                            map_key="SiO2", # color the points by their SiO2 content
                            )

## 4. Interactive line profiles

`mm.interactive_line_profile` lets you draw transects across a map by clicking a start and end point. Each completed transect is extracted and plotted immediately.

**Keybindings:** `r` reset current clicks &nbsp;|&nbsp; `u` undo last transect &nbsp;|&nbsp; `c` clear all &nbsp;|&nbsp; `q`/Esc quit

Access results after quitting:
```python
lp["profiles_df"]     # all profile data concatenated
lp["coordinates_df"]  # transect start/end coordinates
```

In [ ]:
%matplotlib widget

lp = mm.interactive_line_profile(
    result,
    key="SiO2", # key of the oxide map to plot the line profile for
    method="none", # do not apply any smoothing to the line profile. use mean or median (across the pixel width) for a smoothed profile 
    width_px=5, # width of the line profile in pixels
    pixel_size_um=4.0, # pixel size in micrometers
)

### 4a. Filter transects to a single phase

Pass `phase` to mask the background map to pixels of a specific mineral before drawing transects.

In [ ]:
%matplotlib widget

lp_opx = mm.interactive_line_profile(
    result,
    key="SiO2",
    phase="Orthopyroxene",
    method="none",
    width_px=5,
    pixel_size_um=4.0,
)

Return the line profile dataframe. 

In [ ]:
%matplotlib inline
lp["profiles_df"]

## 5. Plot transect locations

Pass the `coordinates_df` from a line profile controller to `mm.plot_locations` to visualize transect positions.

In [ ]:
%matplotlib inline

fig, ax = mm.plot_locations(result, 
                            lp["coordinates_df"],
                            map_key="SiO2"
                            )

## 6. Batch extract profiles from saved transects

`mm.batch_extract_line_profiles` extracts profiles for multiple oxides at once from a saved transect table. Pass `lp["coordinates_df"]` from an interactive session, or any DataFrame with `x0`, `y0`, `x1`, `y1` columns.

Each oxide becomes its own column, making it easy to compare compositions along the same transect.

In [ ]:
%matplotlib inline

profiles_df = mm.batch_extract_line_profiles(
    result,
    transects=lp["coordinates_df"],
    keys=["SiO2", "Al2O3", "FeOt", "MgO", "CaO"],
    method="mean",
    pixel_size_um=4.0,
)

profiles_df

In [ ]:
# Long format — one row per oxide per distance bin, useful for plotting
profiles_df, profiles_long_df = mm.batch_extract_line_profiles(
    result,
    transects=lp["coordinates_df"],
    keys=["SiO2", "MgO", "CaO"],
    method="mean",
    pixel_size_um=4.0,
    return_long=True,
)

profiles_long_df

## 7. Composite component map

`mm.plot_component_composite` overlays continuous solid-solution compositions (e.g. plagioclase An%, olivine Fo%, pyroxene XMg) on top of a categorical phase map. This gives a single figure showing both phase identity and mineral chemistry.

The component maps are computed automatically by `run_map` and stored in `result["component_maps"]`.

In [ ]:
%matplotlib inline

fig, mineral_map, comp_maps = mm.plot_component_composite(
    result,
    title="09g3 Composite",
    pixel_size_um=4.0,
    scalebar_um=50,
)

You can restrict which phases appear and adjust the color limits. `limits_mode="percentile"` clips the colorbar to the 5th–95th percentile of each component, which is useful when a few extreme pixels would otherwise compress the color scale.

In [ ]:
fig, mineral_map, comp_maps = mm.plot_component_composite(
    result,
    title="09g3 Composite (percentile limits)",
    pixel_size_um=4.0,
    scalebar_um=50,
    limits_mode="percentile",
    percentile=(5, 95),
    phases=["Plagioclase", "Orthopyroxene", "Olivine", "Amphibole"],
)